# YOLO26 monocular depth training on DDOS

Fine-tunes `yolo26l-depth.pt` on the depth maps from
[DDOS](https://huggingface.co/datasets/benediktkol/DDOS) (synthetic drone flights, 1280×720).

**Prerequisite:** run `ddos_download.ipynb` first. This notebook doesn't download anything. It
expects the raw data at `datasets/DDOS/data/<split>/<env>/<flight>/{image,depth}/<frame>.png`.

**Why a second dataset folder:** Ultralytics finds each depth target by replacing the `images`
folder in the image path with `depth` (`.../images/train/x.png` → `.../depth/train/x.png`). DDOS
uses `image/` and nests by flight, so section 3 builds `datasets/DDOS-depth/` out of **symlinks**
into the raw download. No pixels are copied, so it takes almost no disk space:
```
datasets/DDOS-depth/
  images/{train,val,test}/<env>_<flight>_<frame>.png  -> DDOS/data/<split>/<env>/<flight>/image/<frame>.png
  depth/{train,val,test}/<env>_<flight>_<frame>.png   -> DDOS/data/<split>/<env>/<flight>/depth/<frame>.png
  data.yaml
```
Frame names include the flight id, so frames from one flight never end up in two splits. The
official DDOS split is kept: `validation` → `val` is used for early stopping, and `test` is only
used once at the end.

**Depth encoding:** DDOS stores uint16 PNGs where 0..65535 maps linearly to 0..100 m. Ultralytics
divides PNG values by `depth_scale`, so `depth_scale = 65535 / 100 = 655.35`. The PNGs are used as-is.
- Pixels at 0 m are treated as invalid by the loss.
- Pixels at the 100 m cap (sky, far background) are trained toward 100 m. The val metrics
  (`max_depth: 100`) leave them out, following the Eigen protocol.

**Pipeline:**
1. Environment checks
2. Configuration
3. Build the symlinked dataset + `data.yaml`
4. Verify the dataset (counts, pairing, decode, depth range)
5. Load model
6. Train
7. Evaluate `best.pt` on the test split
8. Visual check: RGB | ground truth | prediction

## 1. Environment checks

In [ ]:
import os
import torch
import ultralytics
from ultralytics.utils import SETTINGS

print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("datasets_dir:", SETTINGS["datasets_dir"])
print("runs_dir:", SETTINGS["runs_dir"])

## 2. Configuration

These start from the Ultralytics depth fine-tuning recipe (`-depth.pt` weights,
`optimizer=AdamW lr0=1e-4 warmup_bias_lr=1e-4`). Everything else uses the defaults.

| Setting | Default | Here | Why |
|---|---|---|---|
| `MODEL` | – | `yolo26l-depth.pt` | Already in `weights/`. ~30k train frames is enough for L. Use `n`/`s` for faster iterations, `x` (in `yolo2026/weights/`) for best accuracy. |
| `IMGSZ` | 640 | **768** | The resolution the pretrained depth weights were trained at. Frames are 1280×720. |
| `optimizer` / `lr0` / `warmup_bias_lr` | auto / 0.01 / 0.1 | **AdamW / 1e-4 / 1e-4** | Recommended for depth fine-tuning. The default LR is too high for a pretrained depth head. |
| `EPOCHS` / `patience` | 100 / 100 | **60 / 15** | 30k frames per epoch, and neighbouring frames in a flight are very similar, so it converges in fewer epochs. Early stopping does the rest. |
| `flipud` | 0.0 | 0.0 (kept) | The camera faces forward with the sky at the top. A vertical flip would teach upside-down scenes. `fliplr=0.5` stays. |
| mosaic / mixup / cutmix | – | – | Disabled automatically for depth: stitched images would make the depth maps inconsistent. |
| `FRACTION` | 1.0 | 1.0 | Set to e.g. `0.02` for a quick smoke test on 2% of train. |
| `SMOKE_TEST` | – | `True` | Section 6 first runs 1 epoch on 1% of the data. Look at `train_batch0.jpg` / `val_batch0_pred.jpg`, then set it to `False` for the real run. |

In [ ]:
from pathlib import Path

DDOS_DIR = Path(SETTINGS["datasets_dir"]) / "DDOS"          # raw download (ddos_download.ipynb)
OUT_DIR = Path(SETTINGS["datasets_dir"]) / "DDOS-depth"     # symlinked view built below
SPLIT_MAP = {"train": "train", "validation": "val", "test": "test"}   # DDOS split -> YOLO split
ENVIRONMENTS = ["neighbourhood", "park"]

DEPTH_SCALE = 65535 / 100     # uint16 PNG units per meter (0..65535 = 0..100 m)
MAX_DEPTH = 100.0             # val metrics ignore gt >= this (DDOS far plane / sky)

MODEL = str(Path(SETTINGS["weights_dir"]) / "yolo26l-depth.pt")
EPOCHS = 60
IMGSZ = 768
BATCH = 16                    # -1 = AutoBatch (~60% VRAM), 0.8 = 80% VRAM
DEVICE = 0
WORKERS = 16
FRACTION = 1.0
SMOKE_TEST = True             # 1 epoch on 1% of train first; set False for the full run
RUN_NAME = f"yolo26l_ddos_depth_{IMGSZ}"

HYP = dict(
    optimizer="AdamW",
    lr0=1e-4,
    warmup_bias_lr=1e-4,
    patience=15,
    fliplr=0.5,
    flipud=0.0,
    cache=False,     # 30k PNGs at 1280x720 are far too big for a RAM cache
    plots=True,
)
print("Raw data:", DDOS_DIR, "| exists:", DDOS_DIR.exists())
print("YOLO view:", OUT_DIR)
print("Model:", MODEL)

## 3. Build the symlinked dataset

For every frame that has both `image/<n>.png` and `depth/<n>.png`, this creates
`images/<split>/<env>_<flight>_<n>.png` and `depth/<split>/<env>_<flight>_<n>.png` as absolute
symlinks. Frames missing either file are skipped and counted. Safe to re-run: existing links are
kept, and stale links (target removed) are deleted. Then `data.yaml` is written.

In [ ]:
import yaml

if not (DDOS_DIR / "data").exists():
    raise FileNotFoundError(f"{DDOS_DIR / 'data'} not found. Run ddos_download.ipynb first "
                            "(at least the 'image' and 'depth' modalities).")

counts = {}
for src_split, dst_split in SPLIT_MAP.items():
    img_out, dep_out = OUT_DIR / "images" / dst_split, OUT_DIR / "depth" / dst_split
    img_out.mkdir(parents=True, exist_ok=True)
    dep_out.mkdir(parents=True, exist_ok=True)
    for d in (img_out, dep_out):  # drop links whose target vanished
        for link in d.iterdir():
            if link.is_symlink() and not link.exists():
                link.unlink()
    linked = skipped = 0
    for env in ENVIRONMENTS:
        env_dir = DDOS_DIR / "data" / src_split / env
        if not env_dir.exists():
            continue
        for flight in sorted((p for p in env_dir.iterdir() if p.is_dir()), key=lambda p: int(p.name)):
            for img in sorted((flight / "image").glob("*.png")):
                dep = flight / "depth" / img.name
                if not dep.exists():
                    skipped += 1
                    continue
                name = f"{env}_{flight.name}_{img.name}"
                for src, dst in ((img, img_out / name), (dep, dep_out / name)):
                    if not dst.is_symlink():
                        dst.symlink_to(src.resolve())
                linked += 1
    counts[dst_split] = (linked, skipped)

print(f"{'split':6s} {'pairs':>7} {'skipped':>8}")
for s, (n, k) in counts.items():
    print(f"{s:6s} {n:7d} {k:8d}")

DATA_YAML = OUT_DIR / "data.yaml"
data_cfg = {
    "path": str(OUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": {0: "depth"},
    "channels": 3,
    "depth_scale": DEPTH_SCALE,
    "max_depth": MAX_DEPTH,
}
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
print("\nWrote", DATA_YAML)
print(DATA_YAML.read_text())

## 4. Verify the dataset

- `check_det_dataset` makes sure the YAML resolves the way the trainer will see it.
- Every image has a depth file with the same name, so the trainer reports no "missing depth".
- A random sample is decoded through Ultralytics' own `load_depth` (the trainer's loader) to
  check the uint16 format, the size matching the image, and the meter range after `depth_scale`.
  Expect values in (0, 100] m, with a spike at 100 m for sky.

In [ ]:
import random
import cv2
import numpy as np
from ultralytics.data.utils import check_det_dataset, load_depth

data_info = check_det_dataset(str(DATA_YAML))
print("train:", data_info["train"])
print("val:  ", data_info["val"])
print("depth_scale:", data_info.get("depth_scale"), "| max_depth:", data_info.get("max_depth"))

problems = []
for split in SPLIT_MAP.values():
    imgs = {p.name for p in (OUT_DIR / "images" / split).glob("*.png")}
    deps = {p.name for p in (OUT_DIR / "depth" / split).glob("*.png")}
    print(f"{split:6s} images={len(imgs):6d} depth={len(deps):6d}")
    if not imgs and split != "test":
        problems.append(f"{split}: no images")
    if imgs != deps:
        problems.append(f"{split}: {len(imgs - deps)} images without depth, {len(deps - imgs)} depth without image")

SAMPLES = 50
rng = random.Random(0)
mins, maxs, far_frac = [], [], []
for split in ("train", "val"):
    files = sorted((OUT_DIR / "images" / split).glob("*.png"))
    for f in rng.sample(files, min(SAMPLES, len(files))):
        im = cv2.imread(str(f))
        try:
            d = load_depth(OUT_DIR / "depth" / split / f.name, DEPTH_SCALE)
        except Exception as e:
            problems.append(f"{f.name}: {e}")
            continue
        if im is None:
            problems.append(f"{f.name}: image cannot be decoded")
        elif im.shape[:2] != d.shape:
            problems.append(f"{f.name}: image {im.shape[:2]} vs depth {d.shape}")
        valid = d[d > 0.001]
        mins.append(valid.min() if valid.size else np.nan)
        maxs.append(d.max())
        far_frac.append((d >= MAX_DEPTH - 1e-3).mean())

print(f"\nSampled {len(mins)} depth maps: min {np.nanmin(mins):.2f} m, max {np.max(maxs):.2f} m, "
      f"pixels at the {MAX_DEPTH:.0f} m cap: {100 * np.mean(far_frac):.1f}% (excluded from val metrics)")
if np.max(maxs) > MAX_DEPTH + 1e-3:
    problems.append(f"depth exceeds {MAX_DEPTH} m: check DEPTH_SCALE")
for p in problems:
    print("FAIL", p)
if problems:
    raise RuntimeError(f"{len(problems)} dataset problems")
print("Dataset OK")

## 5. Load model

`yolo26l-depth.pt` is the pretrained YOLO26 depth model (trained at 768 px). Fine-tuning starts
from it, not from scratch.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
print("task:", model.task)
model.info()

## 6. Train

With `SMOKE_TEST = True`, this runs **1 epoch on 1% of train** and stops. Check the run folder it
prints:
- `train_batch0.jpg`: depth maps line up with the RGB frames (same flips, no shifts).
- `val_batch0_pred.jpg`: predictions look like scenes, not noise.

Then set `SMOKE_TEST = False` in section 2 and re-run sections 2, 5 and 6.

The full run is long: ~30k frames per epoch at 768 px with the L model. Start it from tmux:
`jupyter nbconvert --to notebook --execute --inplace ddos_depth_train.ipynb`.
The headline val metric is `delta1` (share of pixels within 1.25× of the true depth, higher is
better). `abs_rel` and `rmse` are also reported. After training, Ultralytics automatically fits
a metric-scale calibration on val and keeps it only if it improves the results.

In [ ]:
import multiprocessing as mp

if HYP.get("cache") in (True, "ram") and mp.get_start_method() != "fork":
    print(f"cache={HYP['cache']!r} is unsafe with start method {mp.get_start_method()!r}; using cache=False")
    HYP["cache"] = False

run_args = dict(
    data=str(DATA_YAML),
    epochs=1 if SMOKE_TEST else EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    fraction=0.01 if SMOKE_TEST else FRACTION,
    name=f"{RUN_NAME}_smoke" if SMOKE_TEST else RUN_NAME,   # no project= (see CLAUDE.md)
    **HYP,
)
print(run_args)
results = model.train(**run_args)
SAVE_DIR = Path(model.trainer.save_dir)
BEST = SAVE_DIR / "weights" / "best.pt"
print("Run folder:", SAVE_DIR)
print("best.pt:", BEST)
if SMOKE_TEST:
    print("\nSmoke test done: inspect train_batch0.jpg / val_batch0_pred.jpg above, "
          "then set SMOKE_TEST = False and re-run sections 2, 5, 6.")

## 7. Evaluate on the test split

`best.pt` on the held-out `test` flights. This is the number to report: val was used for early
stopping, so it's slightly optimistic.

In [ ]:
best = YOLO(str(BEST))
test_metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                        name=f"{SAVE_DIR.name}_test")
for k, v in test_metrics.results_dict.items():
    print(f"{k:24s} {v:.4f}")

## 8. Visual check

Random val frames: RGB, ground truth, and prediction on the same color scale (meters, clipped at
`MAX_DEPTH`), plus the absolute error.

In [ ]:
import matplotlib.pyplot as plt

N_SHOW = 4
val_imgs = sorted((OUT_DIR / "images" / "val").glob("*.png"))
picks = random.Random(1).sample(val_imgs, min(N_SHOW, len(val_imgs)))

fig, axes = plt.subplots(len(picks), 4, figsize=(22, 3.4 * len(picks)), squeeze=False)
for row, f in zip(axes, picks):
    rgb = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    gt = load_depth(OUT_DIR / "depth" / "val" / f.name, DEPTH_SCALE)
    r = best.predict(str(f), imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    pred = r.depth.data.squeeze().cpu().numpy() if hasattr(r.depth, "data") else np.asarray(r.depth).squeeze()
    pred = cv2.resize(pred.astype(np.float32), (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_LINEAR)
    err = np.where((gt > 0.001) & (gt < MAX_DEPTH), np.abs(pred - gt), np.nan)
    row[0].imshow(rgb); row[0].set_title(f.stem)
    row[1].imshow(gt, cmap="magma_r", vmin=0, vmax=MAX_DEPTH); row[1].set_title("ground truth (m)")
    im = row[2].imshow(pred, cmap="magma_r", vmin=0, vmax=MAX_DEPTH); row[2].set_title("prediction (m)")
    fig.colorbar(im, ax=row[2], fraction=0.025)
    im = row[3].imshow(err, cmap="viridis", vmin=0, vmax=10); row[3].set_title(f"|error| (m), median {np.nanmedian(err):.2f}")
    fig.colorbar(im, ax=row[3], fraction=0.025)
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()